<a href="https://colab.research.google.com/github/TAUforPython/Graph-MachineLearning/blob/main/examples/hyperbolic_graph_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hyperbolic graph basics / Основы гиперболических графов

**Goal / Цель.** Compute distances in the Poincaré ball and visualize a small tree embedding. / Вычислить расстояния в шаре Пуанкаре и визуализировать эмбеддинг небольшого дерева.

**Prerequisites / Требования.** Basic Python and vectors / основы Python и векторов. **Runtime / Время.** About 1 minute on CPU / около 1 минуты на CPU. **Output / Результат.** Distance checks and a Poincaré-disk plot / проверка расстояний и рисунок диска Пуанкаре.

## 1. Setup / Подготовка

The example needs only NumPy and Matplotlib. A fixed seed makes all results repeatable. / Пример использует только NumPy и Matplotlib. Фиксированный seed обеспечивает воспроизводимость.

In [ ]:
%pip install -q numpy matplotlib

import numpy as np
import matplotlib.pyplot as plt

SEED = 11
rng = np.random.default_rng(SEED)


## 2. Poincaré distance / Расстояние Пуанкаре

For points $u,v$ in the unit ball, / Для точек $u,v$ в единичном шаре:

$$d(u,v)=\operatorname{arcosh}\left(1+2\frac{\|u-v\|^2}{(1-\|u\|^2)(1-\|v\|^2)}\right).$$

The small clamp protects floating-point calculations. Points must stay strictly inside the ball. / Небольшое ограничение защищает вычисления с плавающей точкой. Точки должны находиться строго внутри шара.

In [ ]:
def poincare_distance(u, v, eps=1e-12):
    """Distance in the unit-curvature Poincaré ball."""
    u, v = np.asarray(u, dtype=float), np.asarray(v, dtype=float)
    u2, v2 = u @ u, v @ v
    if u2 >= 1 or v2 >= 1:
        raise ValueError("Points must be strictly inside the unit ball")
    argument = 1 + 2 * np.sum((u - v) ** 2) / ((1 - u2) * (1 - v2))
    return np.arccosh(max(1.0, argument - eps))

origin = np.zeros(2)
for radius in (0.2, 0.5, 0.8, 0.95):
    print(f"radius={radius:.2f}, distance from center={poincare_distance(origin, [radius, 0]):.3f}")

assert np.isclose(poincare_distance(origin, origin), 0.0)
assert poincare_distance(origin, [0.8, 0]) > poincare_distance(origin, [0.5, 0])


## 3. Embed a hierarchy / Размещение иерархии

We place each tree level at a larger radius. Exponentially many leaves gain room near the boundary. This hand-crafted layout teaches geometry; it is not a trained embedding. / Каждый уровень дерева получает больший радиус. Около границы появляется место для экспоненциально растущего числа листьев. Это учебная раскладка, а не обученный эмбеддинг.

In [ ]:
levels = {
    "root": (0, 0.0, 0.0),
    "A": (1, 0.45, 0.0), "B": (1, 0.45, np.pi),
    "A1": (2, 0.78, -0.35), "A2": (2, 0.78, 0.35),
    "B1": (2, 0.78, np.pi - 0.35), "B2": (2, 0.78, np.pi + 0.35),
}
points = {name: np.array([radius * np.cos(angle), radius * np.sin(angle)])
          for name, (_, radius, angle) in levels.items()}
edges = [("root", "A"), ("root", "B"), ("A", "A1"), ("A", "A2"),
         ("B", "B1"), ("B", "B2")]

fig, ax = plt.subplots(figsize=(7, 7))
ax.add_patch(plt.Circle((0, 0), 1, fill=False, linewidth=2, color="black"))
for source, target in edges:
    # Chords aid readability; true Poincaré geodesics are generally circular arcs.
    xy = np.vstack([points[source], points[target]])
    ax.plot(xy[:, 0], xy[:, 1], color="#888888", zorder=1)
for name, point in points.items():
    ax.scatter(*point, s=350, color="#3568b8", zorder=2)
    ax.text(*point, name, ha="center", va="center", color="white", zorder=3)
ax.set(xlim=(-1.05, 1.05), ylim=(-1.05, 1.05), aspect="equal",
       title="A hierarchy inside the Poincaré disk")
ax.axis("off")
plt.show()

for source, target in edges:
    print(f"{source:>4} → {target:<2}: {poincare_distance(points[source], points[target]):.3f}")
assert all(np.linalg.norm(point) < 1 for point in points.values())


## Interpretation / Интерпретация

Euclidean radii are bounded by 1, but hyperbolic distance to the boundary is unbounded. The straight edge chords above are visual aids, not generally geodesics. / Евклидовы радиусы ограничены единицей, но гиперболическое расстояние до границы не ограничено. Прямые рёбра на рисунке — визуальная подсказка, а не геодезические.

Continue with the [bilingual theory](../theory/hyperbolic-graphs/README.md) / Продолжите с [двуязычной теорией](../theory/hyperbolic-graphs/README.md).